In [1]:
!pip install -U spacy
!pip install spacy-transformers
!pip install PyMuPDF

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 102.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.3.7
    Uninstalling huggingface_hub-1.3.7:
      Successfully uninstalled huggingface_hub-1.3.7
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled tra

In [10]:
!git clone https://github.com/yashasvi-shukl/NER-Resume_parser

Cloning into 'NER-Resume_parser'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 22 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 1.37 MiB | 4.94 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [2]:
# Importing required libraries

import spacy
from spacy.tokens import DocBin
from tqdm import tqdm
import json
from spacy import displacy
import sys, fitz

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
spacy.__version__

'3.8.11'

In [12]:
# Loading JSON file (dataset)
cv_data = json.load(open("/content/NER-Resume_parser/CV-Parsing-using-Spacy-3/training/train_data.json"))

In [13]:
len(cv_data)

200

In [15]:
# Setting up configuration file
!python -m spacy init fill-config /content/NER-Resume_parser/CV-Parsing-using-Spacy-3/training/config.cfg CV-Parsing-using-Spacy-3/data/training/config.cfg

✔ Auto-filled config with all values
✔ Saved config
CV-Parsing-using-Spacy-3/data/training/config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [16]:
def get_spacy_doc(file, data):
    nlp = spacy.blank('en')
    db = DocBin()

    for text, annot in tqdm(data):
        doc = nlp.make_doc(text)
        annot = annot['entities']

        ents = []
        entity_indices = []

        # below code will help to identify if there is any overlapping entity. If yes, then we'll keep only 1st entity and skip other
        for start, end , label in annot:
            skip_entitiy = False
            for idx in range(start, end):
                if idx in entity_indices:
                    skip_entitiy = True
                    break
            if skip_entitiy:
                continue


            entity_indices = entity_indices + list(range(start, end))

            #Getting span of data
            try:
                span = doc.char_span(start, end, label = label, alignment_mode='strict')
            except:
                continue

            # If the given index span has no value the we'll add those data into error.txt file
            if span is None:
                err_data = str([start, end]) + "    " + str(text) +'\n'
                file.write(err_data)

            else:
                ents.append(span)


        try:
          # Setting up custom entities and adding it to docbin object
            doc.ents = ents
            db.add(doc)

        except:
            pass

    return db


In [17]:
# Creating train and test dataset

from sklearn.model_selection import train_test_split
train, test = train_test_split(cv_data, test_size = 0.2)

In [18]:
len(train), len(test)

(160, 40)

In [19]:
file = open('error.txt', 'w', encoding='utf-8')

#Extracting and saving spacy file to the disk for training purpose
db = get_spacy_doc(file, train)
db.to_disk('train_data.spacy')

db = get_spacy_doc(file, test)
db.to_disk('test_data.spacy')


file.close()

100%|██████████| 40/40 [00:00<00:00, 66.68it/s]


In [20]:
# Training the custom spacy NER model
!python -m spacy train CV-Parsing-using-Spacy-3/data/training/config.cfg --output ./output --paths.train ./train_data.spacy --paths.dev ./test_data.spacy --gpu-id 0

✔ Created output directory: output
ℹ Saving to output directory: output
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 212kB/s]
config.json: 100% 481/481 [00:00<00:00, 4.94MB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 4.81MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 2.86MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 6.46MB/s]
2026-02-10 07:13:20.428587: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770707600.448466    4068 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770707600.454711    4068 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W

# Model Test

In [21]:
nlp = spacy.load('/content/output/model-best')  # Loading best model

In [ ]:
nlp.get_pipe("ner").labels

('College Name',
 'Companies worked at',
 'Degree',
 'Designation',
 'Email Address',
 'Graduation Year',
 'Location',
 'Name',
 'Skills',
 'UNKNOWN',
 'Years of Experience')

In [22]:
#Setting up colors for custom entities
colors = {
    'College Name' : '#E6B0AA',
    'Companies worked at' : '#AF7AC5',
    'Degree' : '#5499C7',
    'Designation' : '#5DADE2',
    'Email Addresss' : '#48C9B0',
    'Graduation Year' : '#73C6B6',
    'Location' : '#2ECC71',
    'Name' : '#F9E79F',
    'Skills' : '#EB984E',
    'UNKNOWN' : '#D35400',
    'Years of Experience' : '#839192'}


options = {"ents": list(nlp.get_pipe("ner").labels), "colors": colors}

In [29]:
# Reading resume
fname = '/content/LeHoangDang_CV_Fresher.pdf'
doc = fitz.open(fname)

In [30]:
# converting resume into text
text = [page.get_text() for page in doc][0].strip().split()
text = " ".join(text)
text

'Le Hoang Dang Java Developer +84 704 155 153 - danglevt1234@gmail.com - github.com/Damien4822 TARGET With specialized knowledge of Information Systems along with a passion for programming, I would like to find a suitable job so that I can apply and promote my abilities. EDUCATION Ho Chi Minh City University of Technology (HUTECH) Ho Chi Minh City, Vietnam Information Technology Engineer - Information Systems 2020 - 2024 Current GPA: 3.34/4.0 TECHNICAL SKILLS Languages: C#, Java, HTML, CSS, JavaScript Frameworks: Spring (Spring Boot, Spring MVC), ASP.NET (.NET Core, MVC), Bootstrap, ReactJS Databases: Microsoft SQL, MySQL, Oracle, SQLite Other Skills: Systems Analysis and Designs, Database Design English: TOEIC 925 (2024) WORK EXPERIENCE Industry Internship - Java Backend Intern TMA Solutions, Dist. 12, Ho Chi Minh City July 2023 - October 2023 • Designed and developed a Progressive Web App (PWA) that allow users to create, read and changing contents of docx files • Using Java for hand

In [31]:
# converting text into spacy doc and then printing the extracted entities

doc = nlp(text)
text_list = []

for ent in doc.ents:
  if ent.text not in text_list:
    print(ent.text, '\t\t\t\t\t', ent.label_)
    text_list.append(ent.text)

Le Hoang 					 Name
Java Developer 					 Designation
Ho Chi Minh City University of Technology (HUTECH) 					 College Name
Ho Chi Minh City 					 College Name
Languages: C#, Java, HTML, CSS, JavaScript Frameworks: Spring (Spring Boot, Spring MVC), ASP.NET (.NET Core, MVC), Bootstrap, ReactJS Databases: Microsoft SQL, MySQL, Oracle, SQLite Other Skills: Systems Analysis and Designs 					 Skills
Oracle 					 Companies worked at


In [32]:
displacy.render(doc, style="ent", jupyter = True, options = options)